# 06 · Abstraction: Abstract Base Classes (ABC) & Protocols 🆕

**Module 06 – Object-Oriented Programming** · Estimated time: 40 min

> ❗ Neither source covers this properly. The bootcamp uses `raise NotImplementedError`, and the handbook describes abstraction only in words. This lesson shows the real tools.

### 🎯 What you will learn
- Abstraction: defining *what* must be done, not *how*
- `abc.ABC` and `@abstractmethod`: enforce that subclasses implement methods
- Abstract properties
- `collections.abc`: get many methods for free by implementing a few
- `typing.Protocol`: "static duck typing" without inheritance
- When to use which

## 1. The problem with `raise NotImplementedError`

In [1]:
class Shape:
    def area(self):
        raise NotImplementedError("Subclass must implement area()")

class Square(Shape):
    def __init__(self, side):
        self.side = side
    # forgot to implement area()!

sq = Square(3)               # no error yet...
try:
    sq.area()                # ...error only when the method is finally called
except NotImplementedError as e:
    print("Too late:", e)

Too late: Subclass must implement area()


## 2. ✅ Abstract base classes with `abc`

A class that inherits from `ABC` and has `@abstractmethod`s **cannot be instantiated**, and neither can any subclass that hasn't implemented all of them. Errors appear immediately, at creation.

In [2]:
from abc import ABC, abstractmethod
import math

class Shape(ABC):
    @abstractmethod
    def area(self) -> float:
        """Return the area of the shape."""

    @abstractmethod
    def perimeter(self) -> float: ...

    def describe(self) -> str:                  # concrete method, shared by all
        return f"{type(self).__name__}: area={self.area():.2f}, perimeter={self.perimeter():.2f}"

class Circle(Shape):
    def __init__(self, r):
        self.r = r
    def area(self):
        return math.pi * self.r ** 2
    def perimeter(self):
        return 2 * math.pi * self.r

class Rectangle(Shape):
    def __init__(self, w, h):
        self.w, self.h = w, h
    def area(self):
        return self.w * self.h
    def perimeter(self):
        return 2 * (self.w + self.h)

for s in [Circle(1), Rectangle(2, 3)]:
    print(s.describe())

Circle: area=3.14, perimeter=6.28
Rectangle: area=6.00, perimeter=10.00


In [3]:
Shape()            # ❌ TypeError: Can't instantiate abstract class Shape

TypeError: Can't instantiate abstract class Shape without an implementation for abstract methods 'area', 'perimeter'

In [4]:
class Square(Shape):
    def area(self):
        return 4
    # perimeter missing

Square()           # ❌ TypeError, raised immediately

TypeError: Can't instantiate abstract class Square without an implementation for abstract method 'perimeter'

### Abstract properties

In [5]:
class Employee(ABC):
    def __init__(self, name):
        self.name = name

    @property
    @abstractmethod
    def salary(self) -> float: ...

class Intern(Employee):
    @property
    def salary(self):
        return 15_000

print(Intern("Asha").salary)

15000


### A practical example: payment processors

In [6]:
class PaymentProcessor(ABC):
    @abstractmethod
    def pay(self, amount: float) -> str: ...

    def checkout(self, amount):              # template method
        if amount <= 0:
            raise ValueError("amount must be positive")
        return self.pay(amount)

class UPI(PaymentProcessor):
    def __init__(self, upi_id):
        self.upi_id = upi_id
    def pay(self, amount):
        return f"Paid ₹{amount} via UPI ({self.upi_id})"

class Card(PaymentProcessor):
    def __init__(self, last4):
        self.last4 = last4
    def pay(self, amount):
        return f"Charged ₹{amount} to card ****{self.last4}"

for method in [UPI("asha@okbank"), Card("4242")]:
    print(method.checkout(499))

Paid ₹499 via UPI (asha@okbank)
Charged ₹499 to card ****4242


## 3. `collections.abc`: implement a few methods, get the rest free

In [7]:
from collections.abc import Sequence, MutableMapping

class Countdown(Sequence):
    def __init__(self, start):
        self.start = start
    def __len__(self):
        return self.start
    def __getitem__(self, i):
        if not 0 <= i < self.start:
            raise IndexError(i)
        return self.start - i

c = Countdown(5)
print(list(c), 3 in c, c.index(2), c.count(1), list(reversed(c)))   # index/count/in/reversed for free

[5, 4, 3, 2, 1] True 3 1 [1, 2, 3, 4, 5]


In [8]:
class LowerDict(MutableMapping):
    """A dict whose keys are case-insensitive."""
    def __init__(self, *args, **kwargs):
        self._d = {}
        self.update(*args, **kwargs)           # update() comes free from MutableMapping
    def __getitem__(self, k): return self._d[k.lower()]
    def __setitem__(self, k, v): self._d[k.lower()] = v
    def __delitem__(self, k): del self._d[k.lower()]
    def __iter__(self): return iter(self._d)
    def __len__(self): return len(self._d)

h = LowerDict({"Content-Type": "json"})
print(h["CONTENT-TYPE"], "content-type" in h, h.get("missing", "-"), dict(h.items()))

json True - {'content-type': 'json'}


## 4. 🆕 `typing.Protocol`: structural typing ("static duck typing")

An ABC requires classes to **inherit** from it. A `Protocol` only requires them to **have the right methods**. Type checkers verify this, so no inheritance is needed. Great for code you don't control.

In [9]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class SupportsClose(Protocol):
    def close(self) -> None: ...

class File:
    def close(self) -> None:
        print("file closed")

class Socket:
    def close(self) -> None:
        print("socket closed")

def shutdown(resources: list[SupportsClose]) -> None:
    for r in resources:
        r.close()

shutdown([File(), Socket()])                       # neither inherits from SupportsClose!
print(isinstance(File(), SupportsClose))           # works thanks to @runtime_checkable
print(isinstance("text", SupportsClose))

file closed
socket closed
True
False


## 5. Which should I use?

| Tool | Use when |
|---|---|
| Plain duck typing | small scripts, simple code |
| `ABC` + `@abstractmethod` | you own the class hierarchy and want to **force** subclasses to implement methods; share concrete code |
| `collections.abc` base classes | building custom containers |
| `Protocol` | you want type-checked "anything with method X", without inheritance |

## 📝 Summary
- Abstraction = define an interface, hide the implementation.
- `ABC` + `@abstractmethod` prevent instantiating incomplete classes.
- `collections.abc` gives mixin methods for free.
- `Protocol` describes required methods structurally; checked by mypy/pyright.

## 🏋️ Practice
1. Create an abstract `Notifier` with `send(message)`; implement `EmailNotifier` and `SMSNotifier`; send a message through a list of notifiers.
2. Create a `Drawable` Protocol with `draw() -> str`, and a function `render(items: list[Drawable])`.
3. Implement a `Sequence` subclass `Squares(n)` that returns `i*i` for each index.

<details><summary>Show solutions</summary>

```python
from abc import ABC, abstractmethod
from typing import Protocol
from collections.abc import Sequence

class Notifier(ABC):
    @abstractmethod
    def send(self, message: str) -> str: ...
class EmailNotifier(Notifier):
    def send(self, message): return f"Email: {message}"
class SMSNotifier(Notifier):
    def send(self, message): return f"SMS: {message}"
for n in [EmailNotifier(), SMSNotifier()]:
    print(n.send("Hello"))

class Drawable(Protocol):
    def draw(self) -> str: ...
def render(items: list[Drawable]) -> None:
    for i in items:
        print(i.draw())

class Squares(Sequence):
    def __init__(self, n): self.n = n
    def __len__(self): return self.n
    def __getitem__(self, i):
        if not 0 <= i < self.n: raise IndexError
        return i * i
```
</details>